# Entity Ruler - rule based matching

This notebook explores a rule based skill extraction with Entity Ruler.

Starting from the categories included in the taxonomi, skills were collected and refined in dictionaries.
The current list of skills is stored in `src/skills.py` in three different tiers:  
- **Tier 1**: words and phrases matched in any case ("Docker") 

- **Tier 2**: words matched with the exact case, short and ambiguous ("R")

- **Tier 3**: custom aliases mapped to a canonical skill

This structure is more mantainable than writing single rules and allows to automatically build patterns (Entity Ruler rules) once the pipeline is called with `build_nlp`.

Entity ruler is tested on the validation set and the rules refined during the exploration. 

Validation shows two important and known limits for this rule based solution: 
- most of the mistakes are **false negatives**: Entity Ruler misses many skills because they are not written in the skills dictionary. 
- on the other sides there are a few **false positives** because Entity Ruler matches tokens without the flexibility of understanding the context.

In [1]:
import json
import pandas as pd
import textwrap
from collections import Counter

from src.skills import TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM
from src.models import build_nlp, entity_ruler_extract_skills
from src.evaluation import (
    build_alias_to_canonical, 
    normalize_all_ads, 
    calculate_micro_metrics, 
    calculate_macro_metrics, 
    create_error_analysis,
    summarize_errors,
    errors_to_dataframe,
    build_canonical_to_aliases,
    review_ad,
)

In [2]:
with open("../data/train_set.json", "r", encoding="utf-8") as file:
    train_set = json.load(file)

train_set_df = pd.DataFrame(train_set)
train_set_df.head(3)

,id,title,employer,municipality,region,occupation,published,description
0,31266388,Chefsarkitekt – samla ihop och utveckla vår en...,REGERINGSKANSLIET,Stockholm,Stockholms län,Enterprise Architect,2026-07-10T15:15:55,"Digitaliseringsavdelningen, Sektionen för Stra..."
1,30760800,Senior Solutions Architect SAP,Cepheid AB,Solna,Stockholms län,Systemarkitekt,2026-03-18T00:06:15,"At Cepheid, one of Danaher’s 15+ operating com..."
2,30770777,Ingenjör med inriktning mot tillgänglighetsanalys,Försvarets Materielverk,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2026-03-19T13:11:24,Har du erfarenhet av analysarbete och komplexa...


In [3]:
# Building pipeline
nlp = build_nlp()

ruler = nlp.get_pipe("entity_ruler")

# Show how many patterns
print(len(ruler.patterns), "patterns")

852 patterns


The example shows that the rules catch strictly what is written in the skill dictionaries and that is a strong limit due to the context and the inflection of expressions:  
**enterprisearkitektur, kravanalys, integrationsarkitektur, informationsmodeller, molnplattformar, molnstrategier, multi-cloud, dataplattformar**  
are hard skills, but my dictionaries are limited to other forms of these expressions and therefore miss them.

In [4]:
# Checking on an example - parts taken from training set
ad = "Vi moderniserar vår enterprisearkitektur, inför generativ AI, skalar upp molnplattformar och bygger en datadriven organisation. " \
"Det är stort, komplext och samhällsviktigt. " \
"Nu söker vi en chefsarkitekt som vill ta ett helhetsgrepp om arkitekturen och leda riktningen framåt. " \
"Som chefsarkitekt har du en ledande roll, men är inte linjechef. " \
"Ditt uppdrag: Ha huvudansvaret för all arkitektur – verksamhets-, informations-, applikations- och teknisk arkitektur. " \
"Leda och samordna EA‑arbetet, säkerställa struktur, riktning och kvalitet. " \
"Säkerställa att arkitekturen stödjer verksamhetens mål, strategier och förändringsinitiativ. Driva fram gemensamma principer, riktlinjer och målarkitekturer. " \
"Erfarenhet av verksamhetsdriven utveckling och kravanalys. " \
"God förståelse för moderna tekniska plattformar, integrationsarkitektur och informationsmodeller. " \
"Kunskap om arkitekturmetoder som TOGAF, ArchiMate eller liknande. " \
"Erfarenhet av AI-relaterad arkitektur, dataplattformar eller molnstrategier. " \
"Erfarenhet av hybrid- eller multi-cloudlösningar."

print(entity_ruler_extract_skills(ad, nlp))

['ArchiMate', 'Artificial Intelligence', 'Kravhantering', 'TOGAF-certifiering']


## The Entity Ruler pipeline on the entire training set

In [5]:
train_set_df["skills"] = train_set_df["description"].apply(lambda text:entity_ruler_extract_skills(text, nlp))

In [6]:
train_set_df[["description", "skills"]].head(10)

,description,skills
0,"Digitaliseringsavdelningen, Sektionen för Stra...","[ArchiMate, Artificial Intelligence, English, ..."
1,"At Cepheid, one of Danaher’s 15+ operating com...","[Kanban, SAP]"
2,Har du erfarenhet av analysarbete och komplexa...,"[English, Swedish]"
3,About Justera Group Justera Group AB is a lead...,"[Amazon Web Services, Automation, English, Swe..."
4,Vill du ta en ledande roll i att forma vår fra...,[Service Integration]
5,About the Company Avaron helps you find assign...,[Continuous delivery]
6,"At Ubiquiti Inc., we create technology platfor...",[DevOps]
7,"Myndigheten för civilt försvar är den ledande,...","[English, RAN, Swedish]"
8,Är du en av de vassaste på KTH som tar examen ...,"[Artificial Intelligence, Github, Machine lear..."
9,Om företaget Avaron AB är ett växande konsultb...,"[API-design, Data modeling, Data modelling, En..."


In [7]:
# How many skills per add?
train_set_df["n_skills"] = train_set_df["skills"].str.len()
print(train_set_df["n_skills"].describe())
print()

# Most common skills across all ads
skill_counts = train_set_df["skills"].explode().value_counts()
print(skill_counts.head(30))

count    2606.000000
mean        5.102840
std         3.678356
min         0.000000
25%         2.000000
50%         4.500000
75%         7.000000
max        24.000000
Name: n_skills, dtype: float64

skills
Swedish                                      1221
English                                       833
Artificial Intelligence                       485
Kravhantering                                 404
Agile                                         379
Automation                                    369
Windows Azure                                 326
Digitalisering                                314
Integrationer                                 309
IT-miljöer                                    295
Python                                        249
API-design                                    244
Systemutveckling                              244
Machine learning                              224
SQL                                           219
Kravarbete                                 

### Checking empty matches

In some cases Entity Ruler could not match any of its patterns to the ad descriptions. The main causes are:  
- Some ads are simply too short and don't contain any real hard skill (so the ruler behaves correctly)
- The actual hard skills are not listed in the rules
- The rules are not flexible enough to catch loose skill patterns in the unstructured text 

In [8]:
empty_matches = train_set_df.loc[
    train_set_df["skills"].apply(len) == 0,
    ["id", "description", "skills"]
]

print(f"{len(empty_matches)} of {len(train_set_df)} ads have no matches")
empty_matches.head(5)

181 of 2606 ads have no matches


,id,description,skills
10,30696991,About the Company Avaron AB is a growing consu...,[]
29,30995743,We are looking for an experienced Requirement ...,[]
79,30321971,Om jobbet Capgemini strävar efter att vara en ...,[]
92,31431746,Veritaz is a leading IT staffing solutions pro...,[]
106,31098818,In this role you will tackle complex challenge...,[]


In [9]:
def show_ad(text, width=100):
    for paragraph in text.splitlines():
        print(textwrap.fill(paragraph, width=width) if paragraph.strip() else "")

show_ad(empty_matches.iloc[0]["description"])

About the Company Avaron AB is a growing consultancy that matches your expertise with the market's
most interesting assignments, offering a platform where your professional development is central.
About the Assignment You will join a next-generation intranet program within Strategy &
Communications, focused on creating a modern, personalised and scalable digital workplace that
strengthens the employee experience.


In [10]:
# Checks characters before and after a specific detected skill
skill = "Digitalisering"
skill_ads = train_set_df[train_set_df["skills"].apply(lambda skills: skill in skills)]

for _, row in skill_ads.head(5).iterrows():
    print("TITLE:", row["title"])
    text = row["description"]

    for ent in nlp(text).ents:
        if ent.ent_id_ == skill:
            start = ent.start_char
            print(ent.text, "->", ent.ent_id_)
            print(text[max(0, start - 50): start + 50])
            print("-" * 80)

TITLE: Enterprise Architect  Technology Modernization & Innovation
digital transformation -> Digitalisering
ions that directly shape our future and drive our digital transformation. Your mission As Enterprise
--------------------------------------------------------------------------------
TITLE: Senior Ekonomikonsult Microsoft Dynamics 365 Business Central
digitalisering -> Digitalisering
itala samarbetsverktyg kan vi erbjuda en komplett digitalisering av kundernas kärnverksamhet. Fellow
--------------------------------------------------------------------------------
TITLE: IT-arkitekt till Höglandets IT
digital transformation -> Digitalisering
 långsiktiga teknikval, hållbara arkitekturer och digital transformation bidrar du till att skapa fö
--------------------------------------------------------------------------------
digitalisering -> Digitalisering
likvärdig. Du har erfarenhet av arkitekturarbete, digitalisering och komplexa IT-miljöer där du bidr
-------------------------------

## Validation

In [11]:
with open("../data/labeled_val_set.json", "r", encoding="utf-8") as file:
    val_df = pd.DataFrame(json.load(file))

print("Ads in validation set: ", len(val_df))
val_df.head(3)

Ads in validation set:  48


,id,description,skills,spans
0,31398837,Om företaget Siemens Energy möter den ökade ef...,"[Alteryx-baserade dataintegrationslösning, Des...","[[652, 661, Snowflake], [692, 714, Snowflake-a..."
1,30714137,This position combines deep technical expertis...,"[Architecture Competencies, Automotive Afterma...","[[585, 615, Single Point of Contact (SPOC)], [..."
2,31553577,Etteplan är ett ingenjörsföretag med över 4 00...,"[DOORS, Kravanalys, Planering och dokumentatio...","[[764, 774, Kravanalys], [776, 791, kravnedbry..."


In [12]:
val_df = val_df.drop(columns="spans")
val_df = val_df.rename(columns={"skills": "gold_raw"}) 
val_df["pred_raw"] = val_df["description"].apply(lambda text:entity_ruler_extract_skills(text, nlp))

val_df.head()

,id,description,gold_raw,pred_raw
0,31398837,Om företaget Siemens Energy möter den ökade ef...,"[Alteryx-baserade dataintegrationslösning, Des...","[Data modelling, English, Snowflake, dbt]"
1,30714137,This position combines deep technical expertis...,"[Architecture Competencies, Automotive Afterma...","[English, System integreration]"
2,31553577,Etteplan är ett ingenjörsföretag med över 4 00...,"[DOORS, Kravanalys, Planering och dokumentatio...","[DOORS, English, Kravhantering, Swedish]"
3,31349669,Om oss På Techrytera AB börjar rekryteringen m...,"[AI, Azure, Microsoft 365, SharePoint-kompeten...","[Artificial Intelligence, Information Architec..."
4,30472352,Om företaget Avaron AB är ett växande konsultb...,[Dokumentera resultat och presentera slutsatse...,[Swedish]


In [13]:
alias_to_canonical = build_alias_to_canonical(TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM)

gold_raw = dict(zip(val_df["id"], val_df["gold_raw"]))
pred_raw = dict(zip(val_df["id"], val_df["pred_raw"]))

# convert all ads, for gold and predictions
gold = normalize_all_ads(gold_raw, alias_to_canonical)
pred = normalize_all_ads(pred_raw, alias_to_canonical)

In [14]:
micro_metrics = calculate_micro_metrics(gold, pred)
macro_metrics = calculate_macro_metrics(gold, pred)

comparison = pd.DataFrame({"micro": micro_metrics, "macro": macro_metrics})

comparison.head(3)

,micro,macro
precision,0.787,0.745
recall,0.257,0.244
f1,0.388,0.353


In [21]:
# Most frequently missed skills
fn_counter = Counter(s for i in gold for s in gold[i] - pred[i])
print(fn_counter.most_common(20))

[('verifiering', 3), ('informationssäkerhet', 3), ('implementation', 3), ('lösningsdesign', 3), ('säkerhetsanalyser', 2), ('strukturerad dokumentation', 2), ('konfigurations- och ändringshantering', 2), ('spårbarhet mellan krav och tekniska lösningar', 2), ('säkerhetsdokumentation', 2), ('Planering och dokumentation av verifiering', 2), ('systemsäkerhet', 2), ('systemarbete', 2), ('tekniska analyser', 2), ('integrations', 2), ('säkerhet', 2), ('produktutveckling', 2), ('identitet', 2), ('observability', 2), ('nätverk', 2), ('Logic Apps', 2)]


### Error analysis from validation

In [16]:
errors = create_error_analysis(gold, pred)

summary = summarize_errors(errors, alias_to_canonical)

In [17]:
print("How many misses are dictionary gaps:\n", summary["fn_types"]) 
print("\nThe Entity Ruler real mistakes:\n", summary["top_fp"])

How many misses are dictionary gaps:
 {'not in dictionary': 680, 'in dictionary, not matched': 2}

The Entity Ruler real mistakes:
 [('Artificial Intelligence', 7), ('IT-miljöer', 6), ('SAP', 3), ('Dataanalys', 3), ('Kravhantering', 3), ('Continuous delivery', 2), ('Github', 2), ('Logic', 2), ('Windows Azure', 2), ('Digitalisering', 2), ('System integreration', 2), ('Systemarkitektur', 2), ('CI/CD', 2), ('Machine learning', 2), ('Integrationer', 2), ('Change management', 2), ('Systemutveckling', 2), ('Windows', 1), ('Swedish', 1), ('General Data Protection Regulation (GDPR)', 1)]


In [18]:
errs = errors_to_dataframe(errors)
errs.head(10) 

,id,false_positives,false_negatives,n_fp,n_fn
0,31273786,[],"[Architecture Decision Records, CORE, Data Map...",0,51
1,31079160,"[IT-miljöer, Infrastructure as Code]","[AI-infrastruktur, Designa säkra och skalbara ...",2,24
2,30940958,"[Continuous delivery, Github, IT-miljöer, Logi...","[Automate workflows, Azure Functions, Code, Co...",5,23
3,31157086,"[Artificial Intelligence, Machine learning]","[ARR, Define the metrics, Model the data, NRR,...",2,23
4,30472655,"[Artificial Intelligence, BTP]","[Arch, BDC, BTP service, Business Process impa...",2,21
5,31398837,[],"[Alteryx-baserade dataintegrationslösning, Des...",0,21
6,31519605,"[Automation, Masterdata, SAP]","[Clean Core, FI, MM, S/4HANA, S/4HANA-transfor...",3,20
7,30703637,"[Artificial Intelligence, ETL/ELT, LLMs]","[AI models and algorithms, Anthropic APIs, Dat...",3,20
8,31209146,[Change management],"[AI-adoption, AI-governance, AI-strategi, Ente...",1,19
9,30533854,[],"[BoM, DWDM, Data Center Interconnect (DCI), De...",0,19


In [20]:
canonical_to_aliases = build_canonical_to_aliases(alias_to_canonical)
review_ad(30940958, val_df, gold, pred, canonical_to_aliases)

gold: ['Active Directory', 'Automate workflows', 'Azure Functions', 'Code', 'Conditional Access policies', 'DevOps', 'Entra ID', 'Event-based integrations', 'GitHub Actions', 'GitOps', 'Graph API', 'Group Policies', 'IAM platforms', 'Infrastructure as Code', 'Logic Apps', 'MFA configuration', 'Microsoft 365', 'Microsoft ecosystem', 'Office SharePoint Server', 'PnP (Patterns and Practices)', 'Power Platform', 'PubSub', 'Purview', 'Python', 'ServiceNow', 'SuccessFactors', 'Teams', 'User experience', 'Windows Powershell', 'compliance', 'deployment', 'enterprise IT environment', 'event-based integrations', 'governance', 'repository management', 'scripting languages', 'security requirements', 'version-controlled workflows']
pred: ['Active Directory', 'Continuous delivery', 'DevOps', 'Entra ID', 'Github', 'Graph API', 'IAM platforms', 'IT-miljöer', 'Infrastructure as Code', 'Logic', 'Microsoft 365', 'Office SharePoint Server', 'PnP (Patterns and Practices)', 'Power Platform', 'PubSub', 'Pyth